# WWEX Stretch Goal: Quantile Delivery Window Prediction


In [ ]:
import os, json, subprocess, hashlib, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

SEED = 42
REPO_URL = 'https://github.com/Break-Through-Tech/WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping.git'
REPO = Path('WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping')

# Use uploaded CSVs if present, otherwise clone the team repo.
pairs = [(Path('/content/train.csv'), Path('/content/test.csv')), (Path('train.csv'), Path('test.csv'))]
local = next(((a, b) for a, b in pairs if a.exists() and b.exists()), None)
if local is None:
    if not (REPO/'data'/'train.csv').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    local = (REPO/'data'/'train.csv', REPO/'data'/'test.csv')
paths = local
raw_train = pd.read_csv(paths[0], dtype={'origin_zip': 'string', 'dest_zip': 'string'})
raw_test = pd.read_csv(paths[1], dtype={'origin_zip': 'string', 'dest_zip': 'string'})
print('Loaded from:', paths[0].parent, '| raw shapes:', raw_train.shape, raw_test.shape)
assert 'transit_days' in raw_train and 'transit_days' not in raw_test

## 1. Prepare the shipment records

We make carrier and service names consistent, format dates and ZIP codes, mark missing measurements, and remove exact duplicate rows.

We also create extra information from existing columns. These new inputs are called **features**. For example, package dimensions give us package volume, and shipment dates tell us whether a shipment falls on a weekend.

We reuse the earlier notebooks' rules to keep the comparison consistent. Two rules still need confirmation: converting unusually large weights as if they were grams, and using **139** to estimate weight from package size.

In [ ]:
CARRIERS = ['Falcon Post','Northstar Parcel','Summit Express','Ironclad Freight','Vela Freight','Cobalt Regional']
SERVICES = ['Ground','Expedited','TwoDay','Overnight','LTL Standard','LTL Guaranteed']
FILL_COLS = ['weight_lbs','length_in','width_in','height_in']

def clean_raw(frame):
    d = frame.copy()
    for col, allowed in [('carrier', CARRIERS), ('service_level', SERVICES)]:
        mapping = {s.lower(): s for s in allowed}
        d[col] = d[col].astype('string').str.strip().str.lower().map(mapping).astype(object)
    d['shipment_mode'] = d['shipment_mode'].astype('string').str.strip().str.lower().astype(object)
    for col in ['origin_zip', 'dest_zip']:
        z = d[col].astype('string').str.strip()
        d[col] = z.where(z.str.fullmatch(r'\d{1,5}', na=False)).str.zfill(5)
    for col in ['order_date', 'ship_date']:
        d[col] = pd.to_datetime(d[col], errors='coerce')
    for col in FILL_COLS:
        d[col] = pd.to_numeric(d[col], errors='coerce')
    d.loc[d.weight_lbs <= 0, 'weight_lbs'] = np.nan
    grams = ((d.shipment_mode.eq('parcel') & (d.weight_lbs > 150)) |
             (d.shipment_mode.eq('ltl_freight') & (d.weight_lbs > 20000))).fillna(False)
    d.loc[grams, 'weight_lbs'] /= 453.592
    for col in FILL_COLS:
        d[col + '_na'] = d[col].isna().astype('int8')
    return d.drop_duplicates().reset_index(drop=True)

def add_features(data):
    d = data.copy()
    d['order_to_ship_days'] = (d.ship_date - d.order_date).dt.days
    d['is_weekend'] = d.ship_dow.isin([5, 6]).astype('int8')
    d['is_peak_season'] = d.ship_month.isin([11, 12]).astype('int8')
    d['month_sin'] = np.sin(2*np.pi*d.ship_month/12)
    d['month_cos'] = np.cos(2*np.pi*d.ship_month/12)
    d['log_distance'] = np.log1p(d.distance_miles.clip(lower=0))
    d['miles_per_promised_day'] = d.distance_miles / d.promised_days.clip(lower=1)
    d['log_weight'] = np.log1p(d.weight_lbs.clip(lower=0))
    d['volume_cuin'] = d.length_in * d.width_in * d.height_in
    d['dim_weight_lbs'] = np.where(d.shipment_mode.eq('parcel').fillna(False), d.volume_cuin/139.0, d.weight_lbs)
    d['billable_weight_lbs'] = d[['weight_lbs', 'dim_weight_lbs']].max(axis=1)
    d['weight_per_pallet'] = d.weight_lbs / d.pallet_count.replace(0, np.nan)
    d['weather_x_congestion'] = d.weather_severity * d.carrier_congestion_index
    d['distance_x_weather'] = d.distance_miles * d.weather_severity
    d['peak_x_congestion'] = d.is_peak_season * d.carrier_congestion_index
    d['carrier_x_service'] = d.carrier.fillna('Unknown').astype(str) + ' | ' + d.service_level.fillna('Unknown').astype(str)
    return d

train = add_features(clean_raw(raw_train)).sort_values('ship_date').reset_index(drop=True)
test_clean = add_features(clean_raw(raw_test))
assert train.ship_date.notna().all() and train.transit_days.notna().all()

EXCLUDED = {'transit_days','delivered_on_time','shipment_id','order_date','ship_date','origin_zip','dest_zip','carrier_cost_usd'}
feats_all = [c for c in train.columns if c not in EXCLUDED]
WEATHER_CONG = {'weather_severity','carrier_congestion_index','weather_x_congestion','distance_x_weather','peak_x_congestion'}
feats_A = [c for c in feats_all if c not in WEATHER_CONG]   # conservative
feats_B = feats_all                                          # includes weather + congestion
FEATURE_SETS = {'A: no weather/congestion': feats_A, 'B: with weather/congestion': feats_B}
print('Cleaned train:', train.shape, '| features A:', len(feats_A), '| features B:', len(feats_B))
print('transit_days is whole days only:', bool(np.allclose(train.transit_days, train.transit_days.round())))
print('train dates:', train.ship_date.min().date(), 'to', train.ship_date.max().date())
print('test  dates:', test_clean.ship_date.min().date(), 'to', test_clean.ship_date.max().date(), '(same period as train: test is an interleaved sample, not the future)')

## 2. Use earlier shipments to learn and later shipments to check

We divide the records into four groups, in date order:

| Group | What we use it for |
|---|---|
| Fit | Teach the models how shipment details relate to delivery time. |
| Calibration | Adjust the range to aim for 80% coverage. |
| Evaluation | Measure how well the adjusted ranges work. |
| Later check | Check results on August–December 2025, including the busy holiday season. |

**Calibration** means using past prediction errors to adjust the range. **Evaluation** means comparing predictions with actual delivery times.

The evaluation and later-check records are not used to train the models or calculate the adjustments in this notebook. However, these dates were examined in earlier work, so they are not a completely new test. The table below shows the dates and number of shipments in each group.

In [ ]:
holdout_date = train.ship_date.quantile(0.8, interpolation='nearest').normalize()
history = train.loc[train.ship_date < holdout_date]
check = train.loc[train.ship_date >= holdout_date].copy()
dates = np.sort(history.ship_date.dt.normalize().unique())
a, b = int(len(dates)*0.60), int(len(dates)*0.80)
fit_data = history.loc[history.ship_date < dates[a]].copy()
cal_data = history.loc[(history.ship_date >= dates[a]) & (history.ship_date < dates[b])].copy()
eval_data = history.loc[history.ship_date >= dates[b]].copy()
assert fit_data.ship_date.max() < cal_data.ship_date.min() < eval_data.ship_date.min() < check.ship_date.min()
for name, f in [('Fit', fit_data), ('Calibration', cal_data), ('Evaluation', eval_data), ('Check (peak season)', check)]:
    print(f'{name:22s}: {len(f):6,d} shipments | {f.ship_date.min().date()} to {f.ship_date.max().date()}')

## 3. Learn early, middle, and late delivery estimates

For each version, we train three models. Each model makes one estimate for every shipment:

| Estimate | Plain meaning |
|---|---|
| P10 | An early estimate: about 10% of delivery times should be at or below this value if the model is well calibrated. |
| P50 | The middle estimate: about half should be at or below it. |
| P90 | A late estimate: about 90% should be at or below it. |

The early and late estimates form the starting delivery range. For example, an early estimate of 4 days and a late estimate of 6 days gives a **4–6 day range**. These are not the fastest and slowest possible delivery times.

Sometimes the estimates come out in the wrong order. We move the range's edges so the middle estimate stays inside and no delivery time is below zero.

Next, we use the calibration records to work out how much to widen each side. This method is called **conformal calibration**. It aims for 80% coverage, but it does not guarantee 80% on future shipments. After repair and widening, the edges should be read as adjusted range limits, not separately verified P10 and P90 estimates.

We keep the earlier model settings. We do not search for new settings here.

In [ ]:
Q = (0.10, 0.50, 0.90)
ALPHA = 0.20   # target: window contains the real value 80% of the time

def make_preprocessor(X):
    cats = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    nums = [c for c in X if c not in cats]
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True))]), nums),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
                          ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cats)])

def fit_quantile_models(feats):
    models = {}
    for q in Q:
        pipe = Pipeline([('prep', make_preprocessor(fit_data[feats])),
                         ('model', HistGradientBoostingRegressor(loss='quantile', quantile=q, max_iter=250,
                                   learning_rate=0.05, max_leaf_nodes=31, random_state=SEED))])
        pipe.fit(fit_data[feats], fit_data.transit_days.to_numpy())
        models[q] = pipe
    return models

def predict_raw(models, frame, feats):
    return {q: models[q].predict(frame[feats]) for q in Q}

def repair(preds):
    # Median-preserving envelope: guarantees lower <= P50 <= upper, never below 0 days.
    A = np.column_stack([preds[q] for q in Q])
    med = np.maximum(0, A[:, 1])
    lo = np.maximum(0, np.minimum(A.min(axis=1), med))
    hi = np.maximum(A.max(axis=1), med)
    assert np.all((0 <= lo) & (lo <= med) & (med <= hi))
    return lo, med, hi

def exact_correction(scores, alpha=ALPHA):
    # Split-conformal order statistic: how much to widen the window on each side.
    scores = np.asarray(scores)
    rank = int(np.ceil((len(scores) + 1) * (1 - alpha)))
    return float(np.sort(scores)[rank - 1])

MODELS, CAL_RAW, GLOBAL_CORR = {}, {}, {}
for name, feats in FEATURE_SETS.items():
    print('Fitting', name, flush=True)
    MODELS[name] = fit_quantile_models(feats)
    cal_raw = predict_raw(MODELS[name], cal_data, feats)
    lo, med, hi = repair(cal_raw)
    y_cal = cal_data.transit_days.to_numpy()
    GLOBAL_CORR[name] = max(0.0, exact_correction(np.maximum(lo - y_cal, y_cal - hi)))
    print(f'   widening applied to each side: {GLOBAL_CORR[name]:.4f} days')

## 4. Check how often deliveries fall inside the range

We compare the predictions with actual delivery times. We also compare them with simple alternatives based on the carrier's promised time. These alternatives are called **baselines**.

For example, if the carrier promises 5 days, a baseline of **promised time ±1 day** gives a range of **4–6 days**.

| Table column | What it means |
|---|---|
| `n` | Number of shipments checked. |
| `coverage` | Share of actual delivery times inside the range. **0.80 means 80%.** |
| `whole_days_in_window` | Number of whole-day delivery times inside the range. A 4–6 day range includes 4, 5, and 6: three values. This is averaged across shipments. |
| `avg_width_days` | Upper limit minus lower limit. A 4–6 day range has a width of 2 days. |
| `P50_MAE` | Average error of the middle estimate, in days. **0.5 means it was off by half a day on average.** Lower is better. |
| `miss_below` | Share of shipments arriving sooner than the range's lower limit. |
| `miss_above` | Share arriving later than its upper limit. |
| `interval_score` | A score that combines range width and penalties for missed deliveries. Lower is better. |

We want a useful balance: ranges that contain roughly 80% of deliveries without being unnecessarily wide. Always compare coverage and width together.

In [ ]:
def window_metrics(y, lo, med, hi):
    whole_days = np.clip(np.floor(hi + 1e-9) - np.ceil(lo - 1e-9) + 1, 0, None)
    pen = 2 / ALPHA
    return dict(n=len(y), coverage=np.mean((lo <= y) & (y <= hi)), whole_days_in_window=whole_days.mean(),
                avg_width_days=np.mean(hi - lo), P50_MAE=mean_absolute_error(y, med),
                miss_below=np.mean(y < lo), miss_above=np.mean(y > hi),
                interval_score=np.mean((hi - lo) + pen*np.maximum(lo - y, 0) + pen*np.maximum(y - hi, 0)))

def calibrated_window(name, frame):
    feats = FEATURE_SETS[name]
    lo, med, hi = repair(predict_raw(MODELS[name], frame, feats))
    c = GLOBAL_CORR[name]
    return np.maximum(0, lo - c), med, hi + c

rows = []
for period, frame in [('Evaluation (Apr-Aug 2025)', eval_data), ('Check period (Aug-Dec 2025)', check)]:
    y = frame.transit_days.to_numpy()
    for name in FEATURE_SETS:
        lo, med, hi = calibrated_window(name, frame)
        rows.append(dict(period=period, method='Model ' + name, **window_metrics(y, lo, med, hi)))
    p = frame.promised_days.to_numpy().astype(float)
    for off in [(-1, 1), (-2, 1), (-2, 2)]:
        rows.append(dict(period=period, method=f'Promised days {off[0]:+d}/{off[1]:+d} (baseline)',
                         **window_metrics(y, p + off[0], p, p + off[1])))
headline = pd.DataFrame(rows).set_index(['period', 'method'])
display(headline.round(4))
print('Naive promised-days point MAE (evaluation period):',
      round(mean_absolute_error(eval_data.transit_days, eval_data.promised_days), 4))

## 5. Check whether results differ between shipment groups

A good overall result can hide a problem for one carrier or service. We check each carrier, shipment type, service level, and month separately. These tables combine the evaluation and later-check periods.

For each group:

- **Coverage** shows how often its deliveries fell inside the range.
- **`ci_low` and `ci_high`** give a rough uncertainty range around that coverage estimate. Small groups usually have wider ranges because there is less information.
- **`avg_width`** shows the average range width in days.
- **`P50_MAE`** shows the average error of the middle estimate.

If 0.80 falls between `ci_low` and `ci_high`, the data do not clearly rule out the 80% target. That does **not** prove the group meets the target. These uncertainty ranges also do not account for shipments being related over time.

In [ ]:
both = pd.concat([eval_data, check])
y_both = both.transit_days.to_numpy()

def wilson(p, n, z=1.96):
    center = (p + z*z/(2*n)) / (1 + z*z/n)
    half = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return center - half, center + half

def subgroup_table(name, col):
    lo, med, hi = calibrated_window(name, both)
    inside = ((lo <= y_both) & (y_both <= hi))
    w = hi - lo
    out = []
    for g, idx in both.groupby(col, dropna=False).indices.items():
        n = len(idx); p = inside[idx].mean(); l, h = wilson(p, n)
        out.append(dict(group=g, n=n, coverage=p, ci_low=l, ci_high=h, avg_width=w[idx].mean(),
                        P50_MAE=np.abs(y_both[idx] - med[idx]).mean()))
    return pd.DataFrame(out).set_index('group').sort_values('coverage')

both = both.assign(ship_month_label=both.ship_date.dt.strftime('%Y-%m'))
SUBGROUPS = {}
for name in FEATURE_SETS:
    for col in ['carrier', 'shipment_mode', 'service_level', 'ship_month_label']:
        t = subgroup_table(name, col); SUBGROUPS[(name, col)] = t
        print(f'\n=== {name} | by {col} ===')
        display(t.round(3))

### Check routes from origin to destination

A **lane** is a shipping route between two locations, such as one city to another.

We repeat the group check for routes with at least **60 shipments**. Routes with fewer records are left out of this comparison because their results are less stable. Even 60 shipments can leave substantial uncertainty.

The output shows the lowest and highest route coverage, the routes with the lowest coverage, and the share whose uncertainty range includes the 80% target.

In [ ]:
# Lanes (origin -> destination): only lanes with at least 60 shipments, otherwise too noisy to judge.
both['lane'] = both.origin_metro.astype(str) + ' -> ' + both.dest_metro.astype(str)
lane_rows = []
for name in FEATURE_SETS:
    t = subgroup_table(name, 'lane'); t = t[t.n >= 60]
    inside_ci = ((t.ci_low <= 0.80) & (0.80 <= t.ci_high)).mean()
    print(f'{name}: {len(t)} lanes with n>=60 | coverage range {t.coverage.min():.2f}-{t.coverage.max():.2f} | '
          f'{inside_ci:.0%} of lanes have a 95% range that includes 80%')
    display(t.head(5).round(3))
    lane_rows.append(t.assign(feature_set=name))
lane_table = pd.concat(lane_rows)

## 6. Try a separate adjustment for each carrier

So far, each version uses the same widening amount for every carrier. Here we try giving each carrier its own adjustment, using only the calibration records.

We need at least **100 calibration shipments** for a carrier-specific adjustment. Otherwise, we use the shared amount. A carrier's adjustment can widen its range further, but cannot make it narrower than the shared adjustment.

This may help Ironclad Freight, whose ranges miss more deliveries than the target allows. The table compares coverage before and after the change, including Ironclad's results.

`share_windows_changed_in_whole_days` tells us how often the adjustment changed which whole-day values fit inside the range. A small decimal change to an edge may not change those whole-day values at all.

This is an optional experiment. The main results and exported test predictions continue to use the shared adjustment. The observed carrier gains are uncertain, so we do not treat this as a proven improvement.

In [ ]:
MIN_CARRIER_CAL = 100
def carrier_adjust(name):
    feats = FEATURE_SETS[name]
    lo, med, hi = repair(predict_raw(MODELS[name], cal_data, feats)); y = cal_data.transit_days.to_numpy()
    s = np.maximum(lo - y, y - hi); g = GLOBAL_CORR[name]; out = {}
    cc = cal_data.carrier.fillna('Unknown').to_numpy()
    for c in np.unique(cc):
        m = cc == c
        local = exact_correction(s[m]) if m.sum() >= MIN_CARRIER_CAL else g
        out[c] = max(g, local)
    return out

rows = []
for name in FEATURE_SETS:
    adj = carrier_adjust(name)
    feats = FEATURE_SETS[name]
    lo, med, hi = repair(predict_raw(MODELS[name], eval_data, feats)); y = eval_data.transit_days.to_numpy()
    adjv = np.array([adj[c] for c in eval_data.carrier.fillna('Unknown')])
    g = GLOBAL_CORR[name]
    lo_g, hi_g = np.maximum(0, lo - g), hi + g
    lo_c, hi_c = np.maximum(0, lo - adjv), hi + adjv
    same_whole_day = ((np.ceil(lo_g-1e-9) == np.ceil(lo_c-1e-9)) & (np.floor(hi_g+1e-9) == np.floor(hi_c+1e-9)))
    ir = eval_data.carrier.eq('Ironclad Freight').to_numpy()
    rows.append(dict(feature_set=name,
        coverage_global=np.mean((lo_g<=y)&(y<=hi_g)), coverage_carrier=np.mean((lo_c<=y)&(y<=hi_c)),
        ironclad_cov_global=np.mean(((lo_g<=y)&(y<=hi_g))[ir]), ironclad_cov_carrier=np.mean(((lo_c<=y)&(y<=hi_c))[ir]),
        share_windows_changed_in_whole_days=1 - same_whole_day.mean()))
carrier_experiment = pd.DataFrame(rows).set_index('feature_set')
display(carrier_experiment.round(4))
print('Only a small share of windows change at all, and Ironclad coverage differences sit inside its sampling noise.')

## 7. View the results in charts

- **Left chart:** compares carriers using both model versions. The dashed line marks the 80% coverage target. These results combine the evaluation and later-check periods.
- **Right chart:** compares the models with the promised-time baselines during the evaluation period. The bars show coverage. The labels show the average number of whole-day values inside each range.

A longer bar means more deliveries fell inside the range. Check the width label too: higher coverage may come from a wider, less useful range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cov = pd.DataFrame({n: SUBGROUPS[(n, 'carrier')].coverage for n in FEATURE_SETS})
cov.plot.bar(ax=axes[0]); axes[0].axhline(0.8, color='black', ls='--', label='80% target')
axes[0].set_ylim(0.6, 1.0); axes[0].set_ylabel('Coverage'); axes[0].set_title('Window coverage by carrier'); axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=30)
h = headline.reset_index()
h = h[h.period.str.startswith('Evaluation')]
axes[1].barh(h.method, h.coverage); axes[1].axvline(0.8, color='black', ls='--')
for i, (c, w) in enumerate(zip(h.coverage, h.whole_days_in_window)):
    axes[1].text(c + 0.005, i, f'{w:.1f} days wide', va='center', fontsize=8)
axes[1].set_xlim(0.3, 1.1); axes[1].set_xlabel('Coverage'); axes[1].set_title('Model window vs promised-days windows')
plt.tight_layout(); plt.show()

## 8. Save the results and delivery predictions

The notebook creates a folder called **`wwex_window_outputs`** containing:

| File | What it contains |
|---|---|
| `headline_results.csv` | Main model and baseline comparisons. |
| `coverage_by_...csv` | Results for carriers, shipment types, services, months, and routes. |
| `carrier_calibration_experiment.csv` | Results from the optional carrier-specific adjustment. |
| `test_delivery_windows.csv` | Lower, middle, and upper predictions for both versions, with one row per shipment ID. |
| `manifest.json` | A record of the dates, inputs, settings, and software versions used. |

The test predictions use the **shared adjustment**, not the optional carrier-specific adjustment. Although the output column names contain P10 and P90, those columns hold the repaired and widened range limits.

`test.csv` has no actual delivery times, so these are predictions only. It covers dates within the training file's overall period, rather than a new future period. The export keeps one row per shipment ID; repeated IDs are reduced to one row.

Download the output files before resetting Colab so you keep a copy.

In [ ]:
OUT = Path('wwex_window_outputs'); OUT.mkdir(exist_ok=True)
headline.to_csv(OUT/'headline_results.csv')
for (name, col), t in SUBGROUPS.items():
    t.to_csv(OUT/f"coverage_by_{col}__{name.split(':')[0]}.csv")
lane_table.to_csv(OUT/'coverage_by_lane.csv')
carrier_experiment.to_csv(OUT/'carrier_calibration_experiment.csv')

test_out = test_clean[['shipment_id']].copy()
for name in FEATURE_SETS:
    tag = name.split(':')[0]
    lo, med, hi = calibrated_window(name, test_clean)
    test_out[f'{tag}_P10_window_low'] = lo; test_out[f'{tag}_P50'] = med; test_out[f'{tag}_P90_window_high'] = hi
test_out = test_out.drop_duplicates('shipment_id')
test_out.to_csv(OUT/'test_delivery_windows.csv', index=False)

manifest = dict(seed=SEED, alpha=ALPHA, widening_days=GLOBAL_CORR,
    periods={n: dict(n=len(d), start=str(d.ship_date.min().date()), end=str(d.ship_date.max().date()))
             for n, d in [('fit', fit_data), ('calibration', cal_data), ('evaluation', eval_data), ('check', check)]},
    features_A=feats_A, features_B=feats_B,
    input_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths},
    versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__))
(OUT/'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Saved to', OUT.resolve(), '| test windows:', len(test_out), 'rows')